# 02 — Threat Modelling

## What This Is
Threat modelling is a structured approach to identifying security requirements by reasoning about adversaries, their goals, and attack paths. It produces a prioritised list of threats and mitigations specific to your system.

## Why It Matters
Threat modelling at design time is 100× cheaper than fixing breaches in production. Microsoft mandates threat modelling for all significant features. It forces explicit security reasoning before code is written — finding architectural flaws that automated tools cannot detect.

## Where the Community Stands
STRIDE (Spoofing, Tampering, Repudiation, Information Disclosure, Denial of Service, Elevation of Privilege) is the dominant framework. PASTA (Process for Attack Simulation and Threat Analysis) is risk-driven. LINDDUN focuses on privacy. Threat modelling as code (Pytm, ThreatSpec) integrates into DevSecOps.

## STRIDE Framework
- **S**poofing: impersonating users or systems
- **T**ampering: modifying data without authorisation
- **R**epudiation: denying actions without audit trail
- **I**nformation Disclosure: exposing sensitive data
- **D**enial of Service: making a system unavailable
- **E**levation of Privilege: gaining higher access than intended

In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Optional

STRIDE_CATEGORIES = {
    'S': ('Spoofing',              'Impersonating something or someone else'),
    'T': ('Tampering',             'Modifying data or code'),
    'R': ('Repudiation',           'Claiming not to have performed an action'),
    'I': ('Information Disclosure','Exposing information to unauthorised parties'),
    'D': ('Denial of Service',     'Preventing access to a system or component'),
    'E': ('Elevation of Privilege','Gaining capabilities without authorisation'),
}

# STRIDE-per-element: which threats apply to which diagram elements
ELEMENT_THREATS = {
    'External Entity': ['S', 'R'],
    'Process':         ['S', 'T', 'R', 'I', 'D', 'E'],
    'Data Store':      ['T', 'I', 'D'],
    'Data Flow':       ['T', 'I', 'D'],
}

for elem, threats in ELEMENT_THREATS.items():
    print(f'{elem}:')
    for t in threats:
        name, desc = STRIDE_CATEGORIES[t]
        print(f'  {t} — {name}: {desc}')

In [ ]:
@dataclass
class ThreatModelElement:
    name:         str
    element_type: str   # External Entity / Process / Data Store / Data Flow
    description:  str

@dataclass
class Threat:
    id:            str
    stride_type:   str
    target:        str
    description:   str
    attack_vector: str
    impact:        str   # Critical/High/Medium/Low
    likelihood:    str   # High/Medium/Low
    mitigation:    str
    status:        str = 'Open'   # Open/Mitigated/Accepted

    def risk_score(self) -> int:
        impact_map    = {'Critical': 4, 'High': 3, 'Medium': 2, 'Low': 1}
        likelihood_map= {'High': 3, 'Medium': 2, 'Low': 1}
        return impact_map.get(self.impact, 0) * likelihood_map.get(self.likelihood, 0)

class ThreatModel:
    def __init__(self, system_name: str):
        self.system_name = system_name
        self.elements: List[ThreatModelElement] = []
        self.threats:  List[Threat] = []
        self._threat_id = 0

    def add_element(self, elem: ThreatModelElement) -> None:
        self.elements.append(elem)

    def add_threat(self, stride: str, target: str, description: str,
                   attack_vector: str, impact: str, likelihood: str,
                   mitigation: str) -> Threat:
        self._threat_id += 1
        t = Threat(f'T{self._threat_id:03d}', stride, target, description,
                   attack_vector, impact, likelihood, mitigation)
        self.threats.append(t)
        return t

    def report(self) -> None:
        sorted_threats = sorted(self.threats, key=lambda t: t.risk_score(), reverse=True)
        print(f'Threat Model: {self.system_name}')
        print(f'Elements: {len(self.elements)}  Threats: {len(self.threats)}')
        print()
        print(f'{"ID":<6} {"STRIDE":<5} {"Target":<20} {"Impact":<10} {"Like":<8} {"Score":<6} {"Status"}')
        print('-' * 75)
        for t in sorted_threats:
            print(f'{t.id:<6} {t.stride_type:<5} {t.target:<20} {t.impact:<10} {t.likelihood:<8} {t.risk_score():<6} {t.status}')

# Model: E-commerce API
model = ThreatModel('E-Commerce REST API')

model.add_element(ThreatModelElement('Mobile App',    'External Entity', 'iOS/Android client'))
model.add_element(ThreatModelElement('API Gateway',   'Process',         'Rate limiting + auth'))
model.add_element(ThreatModelElement('Order Service', 'Process',         'Business logic'))
model.add_element(ThreatModelElement('User DB',       'Data Store',      'PostgreSQL users/orders'))
model.add_element(ThreatModelElement('Payment API',   'External Entity', 'Stripe'))

model.add_threat('S', 'API Gateway',   'JWT token forgery via weak signing key',
                 'Forge JWT with HS256 using known/weak key',
                 'Critical', 'Medium', 'Use RS256 asymmetric signing; rotate keys; validate iss/aud/exp')
model.add_threat('T', 'User DB',       'SQL injection in order history query',
                 'Malicious SQL via unparameterised query',
                 'Critical', 'Medium', 'Parameterised queries; ORM; SAST scan')
model.add_threat('E', 'Order Service', 'IDOR: access other users orders',
                 'Manipulate order_id in API request',
                 'High', 'High', 'Server-side authorisation; check user owns resource')
model.add_threat('I', 'User DB',       'Data breach via excessive data retention',
                 'Leaked DB exposes PII beyond retention policy',
                 'High', 'Medium', 'Data minimisation; encryption at rest; retention policy')
model.add_threat('D', 'API Gateway',   'Rate limit bypass — credential stuffing',
                 'Distributed attack from botnet bypasses per-IP limits',
                 'High', 'High', 'Token bucket per user; CAPTCHA; IP reputation')
model.add_threat('R', 'Order Service', 'Missing audit log for order mutations',
                 'Admin modifies order with no audit trail',
                 'Medium', 'Low', 'Immutable audit log; log user/timestamp/action')

model.report()